# Intelli-Speech — EXP1_CHARACTERIZATION (Experiment 1)

Per-recording acoustic features → speaker-level pattern catalogue for atypical Telugu speech.

**Run only after** `AUDIT_V2_FOLLOWUP.ipynb` (A1 clipping verdicts, A2 transcription scripts,
B typical_controls_v2). Temporal/prosodic features are primary; spectral features are
band-limited and checked for channel confounding. Speakers are the unit of inference
(7 atypical speakers; two have only 5 recordings).

Outputs: `Intelli-Speech-data/analysis_v2/exp1_features_<date>/` (features.csv, analysis/).
**Do not commit this notebook with outputs.** Results are observations about 7 speakers,
not clinical markers.

In [ ]:
# CELL 1 | LOGIN + SYNC + DEPENDENCIES
from huggingface_hub import notebook_login, whoami
try:
    print("Logged in as:", whoami()["name"])
except Exception:
    notebook_login()
import os, subprocess, sys
from pathlib import Path
REPO_DIR = Path("/content/Intelli-Speech")
if not Path("/content/drive/MyDrive").is_dir():
    from google.colab import drive
    drive.mount("/content/drive")
if not REPO_DIR.is_dir():
    subprocess.run(["git", "clone", "https://github.com/niharika9nx/Intelli-Speech.git", str(REPO_DIR)], check=True)
elif not subprocess.run(["git", "-C", str(REPO_DIR), "status", "--short"], capture_output=True, text=True).stdout.strip():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", "main"], check=True)
os.chdir(REPO_DIR); sys.path.insert(0, str(REPO_DIR))
assert Path("scripts/extract_features_exp1.py").is_file(), "Pull the latest code first."
!pip install -q datasets soundfile praat-parselmouth statsmodels "librosa==0.10.2.post1" "numpy<2.5"
DATA_ROOT = "/content/drive/MyDrive/Intelli-Speech-data"
HF_CACHE = f"{DATA_ROOT}/hf_cache"

In [ ]:
# CELL 2 | SMOKE TEST (3 recordings per source, separate folder)
!python scripts/extract_features_exp1.py --data-root "{DATA_ROOT}" --hf-cache "{HF_CACHE}" --limit 3 --output-dir "{DATA_ROOT}/analysis_v2/exp1_smoke" --overwrite

In [ ]:
# CELL 3 | FULL FEATURE EXTRACTION (CPU; ~109 min atypical + controls)
!python scripts/extract_features_exp1.py --data-root "{DATA_ROOT}" --hf-cache "{HF_CACHE}"

In [ ]:
# CELL 4 | SPEAKER-LEVEL ANALYSIS → PATTERN CATALOGUE
import json
FEAT_DIR = sorted(Path(DATA_ROOT, "analysis_v2").glob("exp1_features_*"))[-1]
!python scripts/analyze_exp1.py --features "{FEAT_DIR}/features.csv"

In [ ]:
# CELL 5 | CATALOGUE + PLOTS (pseudonymous speakers)
import pandas as pd, numpy as np, matplotlib.pyplot as plt
cat = pd.read_csv(FEAT_DIR / "analysis" / "catalogue.csv")
cols = ["feature", "family", "verdict", "consistency_k", "consistency_n", "direction", "cliffs_delta", "fdr_q",
        "loso_sign_stable", "icc1_atypical", "max_abs_channel_rho"]
print(cat[[c for c in cols if c in cat]].sort_values(["verdict", "family"]).to_string(index=False))

feats = pd.read_csv(FEAT_DIR / "features.csv", dtype={"speaker_id": str})
feats = feats[feats.get("error").isna()] if "error" in feats else feats
alias = {s: f"S{i+1}" for i, s in enumerate(sorted(feats.loc[feats.group == "atypical", "speaker_id"].unique()))}
top = cat[cat.verdict.isin(["candidate_class1", "speaker_specific"])].feature.tolist()[:8] or cat.feature.tolist()[:8]
control = feats[feats.group == "typical"].groupby("speaker_id")[top].median()
fig, axes = plt.subplots(len(top), 1, figsize=(8, 2.2 * len(top)), squeeze=False)
for ax, f in zip(axes[:, 0], top):
    lo, hi = np.nanpercentile(control[f], [10, 90])
    ax.axhspan(lo, hi, color="0.85", label="control speakers P10–P90")
    atyp = feats[feats.group == "atypical"]
    for i, (spk, sub) in enumerate(sorted(atyp.groupby("speaker_id"), key=lambda kv: alias[kv[0]])):
        ax.scatter(np.full(len(sub), i) + np.random.uniform(-0.15, 0.15, len(sub)), sub[f], s=8, alpha=0.5)
        ax.plot([i - 0.3, i + 0.3], [sub[f].median()] * 2, color="k")
    ax.set_xticks(range(len(alias))); ax.set_xticklabels(sorted(alias.values())); ax.set_title(f, fontsize=9)
fig.tight_layout(); fig.savefig(FEAT_DIR / "analysis" / "top_features_by_speaker.png", dpi=120); plt.show()

Save the notebook and tell Claude. Do not commit it with outputs.

In [ ]:
# CELL 6 | ROBUSTNESS: replicate vs vaani_copy + duration-matched atypical windows (~10 min CPU)
import json
!python scripts/exp1_robustness.py --features "{FEAT_DIR}/features.csv" --hf-cache "{HF_CACHE}"
rob = pd.read_csv(FEAT_DIR / "robustness" / "robustness_summary.csv")
print(rob[[c for c in rob if c.startswith(("feature", "verdict_", "k_", "robust"))]].to_string(index=False))